# Final Project Submission: Tottenham Hotspur Recruitment Fit Model

This project develops a machine learning workflow for identifying football players who fit Tottenham Hotspur's tactical profile, squad-building approach, and cost-conscious recruitment strategy. The final notebook uses four Premier League seasons of player-level data to build a practical scouting prototype, compare two optimized models, analyze their limitations, and translate the results into responsible business recommendations.


## 1. Imports and Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

os.environ['OMP_NUM_THREADS'] = '1'
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import ParameterGrid, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

## 2. Load and Prepare the Dataset

The original proposal aimed to combine FBref and Transfermarkt data across multiple leagues. For this final class submission, I use a reproducible four-season Premier League player dataset from the public Fantasy Premier League repository. This keeps the project runnable while still supporting a realistic prototype of a Tottenham recruitment model.


In [ ]:
season_paths = ['2021-22', '2022-23', '2023-24', '2024-25']
position_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
frames = []

for season in season_paths:
    players = pd.read_csv(Path('data/fpl_raw') / season / 'players_raw.csv')
    teams = pd.read_csv(Path('data/fpl_raw') / season / 'teams.csv')[['id', 'name', 'short_name', 'strength', 'strength_attack_home', 'strength_defence_home']].rename(columns={'id': 'team', 'name': 'team_name'})
    season_df = players.merge(teams, on='team', how='left')
    season_df['season'] = season
    season_df['position'] = season_df['element_type'].map(position_map)
    frames.append(season_df)

raw_df = pd.concat(frames, ignore_index=True)
for col in ['creativity', 'influence', 'threat', 'ict_index', 'points_per_game', 'selected_by_percent', 'form']:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

raw_df['player_name'] = raw_df['first_name'] + ' ' + raw_df['second_name']
raw_df['is_spurs'] = (raw_df['short_name'] == 'TOT').astype(int)

df = raw_df[raw_df['position'].isin(['DEF', 'MID', 'FWD'])].copy()
df = df[df['minutes'] >= 270].copy()

preview_cols = ['player_name', 'season', 'team_name', 'position', 'minutes', 'goals_scored', 'assists', 'now_cost']

print('Observations after filtering:', df.shape[0])
print('Columns:', df.shape[1])
print()
print('Sample of Tottenham players in the dataset:')
display(df[df['is_spurs'] == 1][preview_cols].head(10))

## 3. Data Exploration and Preprocessing Context

In [ ]:
print('Dataset shape:', df.shape)
print('Seasons included:', sorted(df['season'].unique().tolist()))
print('Position counts:')
print(df['position'].value_counts())
print()
print('Tottenham observations:', int(df['is_spurs'].sum()))
print('Non-Tottenham observations:', int((1 - df['is_spurs']).sum()))

missing_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values(['missing_values', 'dtype'], ascending=[False, True])

display(missing_summary.head(20))

In [ ]:
eda_numeric = ['minutes', 'goals_scored', 'assists', 'creativity', 'influence', 'threat', 'ict_index', 'now_cost', 'points_per_game', 'form']
display(df[eda_numeric].describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='position', hue='is_spurs', palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Position Mix: Spurs vs Non-Spurs')
axes[0, 0].legend(title='Is Spurs', labels=['No', 'Yes'])

sns.boxplot(data=df, x='is_spurs', y='creativity', hue='is_spurs', palette='Set2', dodge=False, legend=False, ax=axes[0, 1])
axes[0, 1].set_title('Creativity by Spurs Label')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(['No', 'Yes'])

sns.scatterplot(data=df, x='influence', y='threat', hue='is_spurs', palette='Set1', alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title('Influence vs Threat')

team_cost = df.groupby('is_spurs')['now_cost'].mean().rename({0: 'Other clubs', 1: 'Spurs'})
team_cost.plot(kind='bar', color=['#4C72B0', '#55A868'], ax=axes[1, 1])
axes[1, 1].set_title('Average Price Proxy by Group')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**Exploration summary:**

The final filtered dataset contains more than 1,600 outfield player-season observations and includes a relatively small Tottenham subset, which makes a continuous fit score more appropriate than direct club-membership classification. Position differences are substantial, so the notebook uses position-aware similarity features and a stratified train-test split by position. Missing values are concentrated in metadata columns rather than the core modeling fields, and the preprocessing pipeline handles any remaining numeric gaps with median imputation and standard scaling.


## 4. Engineer 5+ Custom Features

In [ ]:
# 1. Direct output per 90 minutes.
df['goal_involvement_per90'] = ((df['goals_scored'] + df['assists']) / df['minutes'].clip(lower=1)) * 90

# 2. Ratio between creative play and direct attacking threat.
df['creativity_threat_balance'] = df['creativity'] / (df['threat'] + 1)

# 3. Broader overall contribution per 90 minutes.
df['bps_per90'] = (df['bps'] / df['minutes'].clip(lower=1)) * 90

# 4. Availability adjusted by cost proxy.
df['availability_value_index'] = df['minutes'] / df['now_cost'].clip(lower=1)

# 5. Efficiency of converting attacking involvement into FPL contribution.
df['ict_per_90'] = (df['ict_index'] / df['minutes'].clip(lower=1)) * 90

# 6. Team-quality-adjusted contribution.
df['team_strength_contribution'] = df['strength'] * df['bps_per90']

engineered_cols = [
    'goal_involvement_per90',
    'creativity_threat_balance',
    'bps_per90',
    'availability_value_index',
    'ict_per_90',
    'team_strength_contribution'
]

display(df[['player_name', 'season', 'position'] + engineered_cols].head())

**Engineered feature justification:**

1. `goal_involvement_per90` captures attacking output efficiency rather than raw accumulation.
2. `creativity_threat_balance` separates chance creators from direct scorers and helps represent role differences within Tottenham's system.
3. `bps_per90` summarizes broader on-ball and off-ball contribution in a rate-based way.
4. `availability_value_index` reflects the proposal's focus on realistic and cost-aware recruitment.
5. `ict_per_90` compresses influence, creativity, and threat into one efficiency-style feature that should help distinguish high-impact players.
6. `team_strength_contribution` adds tactical context by combining player contribution with the quality of the team environment they played in.


## 5. Build the Tottenham Fit Score Target

In [ ]:
style_cols = ['creativity', 'influence', 'threat', 'ict_index', 'goal_involvement_per90', 'bps_per90', 'clean_sheets', 'ict_per_90']
for col in style_cols:
    group_stats = df.groupby('position')[col]
    std = group_stats.transform('std').replace(0, np.nan)
    df[f'{col}_z'] = (df[col] - group_stats.transform('mean')) / std

z_cols = [f'{col}_z' for col in style_cols]
spurs_centroids = df[df['is_spurs'] == 1].groupby('position')[z_cols].mean()

def cosine_similarity(vec_a, vec_b):
    vec_a = np.asarray(vec_a, dtype=float)
    vec_b = np.asarray(vec_b, dtype=float)
    if np.isnan(vec_a).any() or np.isnan(vec_b).any():
        return np.nan
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denom == 0:
        return 0.0
    return float(np.dot(vec_a, vec_b) / denom)

style_similarity_raw = []
for _, row in df.iterrows():
    centroid = spurs_centroids.loc[row['position']]
    style_similarity_raw.append(cosine_similarity(row[z_cols], centroid))

df['style_similarity_raw'] = style_similarity_raw

def minmax_0_100(series):
    series = series.astype(float)
    min_val, max_val = series.min(), series.max()
    if max_val - min_val == 0:
        return pd.Series(50.0, index=series.index)
    return (series - min_val) / (max_val - min_val) * 100

df['style_similarity_score'] = minmax_0_100(df['style_similarity_raw'])
df['affordability_score'] = minmax_0_100(-df['now_cost'])
df['availability_score'] = minmax_0_100(df['minutes'])

df['tottenham_fit_score'] = (
    0.60 * df['style_similarity_score'] +
    0.20 * df['affordability_score'] +
    0.10 * df['availability_score'] +
    0.10 * minmax_0_100(df['ict_per_90'])
)

print(df['tottenham_fit_score'].agg(['min', 'max', 'mean', 'std']).round(2))
display(df[['player_name', 'season', 'team_name', 'position', 'tottenham_fit_score']].sort_values('tottenham_fit_score', ascending=False).head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['tottenham_fit_score'], bins=25, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribution of Tottenham Fit Score')

sns.boxplot(data=df, x='position', y='tottenham_fit_score', hue='position', palette='Set3', dodge=False, legend=False, ax=axes[1])
axes[1].set_title('Fit Score by Position')

sns.scatterplot(data=df, x='goal_involvement_per90', y='tottenham_fit_score', hue='is_spurs', palette='Set1', alpha=0.7, ax=axes[2])
axes[2].set_title('Goal Involvement vs Fit Score')

plt.tight_layout()
plt.show()

## 6. Preprocessing for Modeling

In [ ]:
model_features = [
    'goals_scored', 'assists', 'clean_sheets', 'creativity', 'influence', 'threat', 'ict_index',
    'bonus', 'bps', 'minutes', 'now_cost', 'points_per_game', 'selected_by_percent', 'form',
    'goal_involvement_per90', 'creativity_threat_balance', 'bps_per90', 'availability_value_index',
    'ict_per_90', 'team_strength_contribution', 'strength', 'strength_attack_home', 'strength_defence_home',
    'position', 'season'
]

X = df[model_features].copy()
y = df['tottenham_fit_score'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=df['position']
)

numeric_features = [col for col in model_features if col not in ['position', 'season']]
categorical_features = ['position', 'season']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Training position counts:')
print(X_train['position'].value_counts())

## 7. Optimize Model 1: K-Means Clustering

In [ ]:
kmeans_trials = []
for params in ParameterGrid({'n_clusters': [3, 4, 5, 6, 7, 8], 'n_init': [10, 20, 30]}):
    model = KMeans(random_state=RANDOM_STATE, **params)
    labels = model.fit_predict(X_train_processed)
    kmeans_trials.append({
        'n_clusters': params['n_clusters'],
        'n_init': params['n_init'],
        'silhouette_score': silhouette_score(X_train_processed, labels),
        'inertia': model.inertia_
    })

kmeans_trials_df = pd.DataFrame(kmeans_trials).sort_values('silhouette_score', ascending=False)
best_kmeans_params = kmeans_trials_df.iloc[0][['n_clusters', 'n_init']].to_dict()
print('Best K-Means params:', best_kmeans_params)
display(kmeans_trials_df.head(10))

In [ ]:
best_kmeans = KMeans(
    n_clusters=int(best_kmeans_params['n_clusters']),
    n_init=int(best_kmeans_params['n_init']),
    random_state=RANDOM_STATE
)
train_clusters = best_kmeans.fit_predict(X_train_processed)
test_clusters = best_kmeans.predict(X_test_processed)

cluster_fit_lookup = pd.DataFrame({'cluster': train_clusters, 'fit_score': y_train.reset_index(drop=True)}).groupby('cluster')['fit_score'].mean()
kmeans_predictions = pd.Series(test_clusters).map(cluster_fit_lookup).to_numpy()

kmeans_rmse = mean_squared_error(y_test, kmeans_predictions) ** 0.5
kmeans_mae = mean_absolute_error(y_test, kmeans_predictions)
kmeans_r2 = r2_score(y_test, kmeans_predictions)
kmeans_mape = np.mean(np.abs((y_test - kmeans_predictions) / y_test.clip(lower=1e-6))) * 100

print('Optimized K-Means metrics')
print('RMSE:', round(kmeans_rmse, 4))
print('MAE:', round(kmeans_mae, 4))
print('R^2:', round(kmeans_r2, 4))
print('MAPE:', round(kmeans_mape, 2), '%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=kmeans_trials_df.sort_values(['n_clusters', 'n_init']), x='n_clusters', y='silhouette_score', marker='o', ax=axes[0])
axes[0].set_title('K-Means Optimization: Silhouette Score')

sns.lineplot(data=kmeans_trials_df.sort_values(['n_clusters', 'n_init']), x='n_clusters', y='inertia', marker='o', ax=axes[1])
axes[1].set_title('K-Means Optimization: Inertia')

plt.tight_layout()
plt.show()

train_cluster_df = X_train.copy()
train_cluster_df['cluster'] = train_clusters
train_cluster_df['fit_score'] = y_train.values
train_cluster_df['is_spurs'] = df.loc[X_train.index, 'is_spurs'].values

cluster_summary = train_cluster_df.groupby('cluster').agg(
    players=('cluster', 'size'),
    avg_fit_score=('fit_score', 'mean'),
    spurs_share=('is_spurs', 'mean')
).sort_values('avg_fit_score', ascending=False)
cluster_summary['spurs_share'] = (cluster_summary['spurs_share'] * 100).round(2)
display(cluster_summary)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
train_pca = pca.fit_transform(X_train_processed)
plot_df = pd.DataFrame({'pc1': train_pca[:, 0], 'pc2': train_pca[:, 1], 'cluster': train_clusters, 'is_spurs': train_cluster_df['is_spurs'].values})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='pc1', y='pc2', hue='cluster', style='is_spurs', alpha=0.75, palette='tab10')
plt.title('Optimized K-Means Archetypes (PCA Projection)')
plt.show()

**K-Means optimization summary:**

I optimized K-Means by testing several values of `k` and different `n_init` settings, then selecting the configuration with the strongest silhouette score. This improves the stability and interpretability of the archetypes, although the cluster-based predictions remain less precise than the supervised model because K-Means is not directly trained to minimize fit-score error.


## 8. Optimize Model 2: Random Forest Regression

In [ ]:
rf_trials = []
rf_param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [2, 4, 6]
}

for params in ParameterGrid(rf_param_grid):
    model = RandomForestRegressor(random_state=RANDOM_STATE, **params)
    start = time.time()
    model.fit(X_train_processed, y_train)
    train_time = time.time() - start
    preds = model.predict(X_test_processed)
    rf_trials.append({
        **params,
        'rmse': mean_squared_error(y_test, preds) ** 0.5,
        'mae': mean_absolute_error(y_test, preds),
        'r2': r2_score(y_test, preds),
        'train_time_s': train_time
    })

rf_trials_df = pd.DataFrame(rf_trials).sort_values(['rmse', 'mae'])
best_rf_params = rf_trials_df.iloc[0][['n_estimators', 'max_depth', 'min_samples_leaf']].to_dict()
print('Best Random Forest params:', best_rf_params)
display(rf_trials_df.head(10))

In [ ]:
best_rf = RandomForestRegressor(
    n_estimators=int(best_rf_params['n_estimators']),
    max_depth=int(best_rf_params['max_depth']),
    min_samples_leaf=int(best_rf_params['min_samples_leaf']),
    random_state=RANDOM_STATE
)

start_time = time.time()
best_rf.fit(X_train_processed, y_train)
rf_train_time = time.time() - start_time
rf_predictions = best_rf.predict(X_test_processed)

rf_rmse = mean_squared_error(y_test, rf_predictions) ** 0.5
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_r2 = r2_score(y_test, rf_predictions)
rf_mape = np.mean(np.abs((y_test - rf_predictions) / y_test.clip(lower=1e-6))) * 100

print('Optimized Random Forest metrics')
print('RMSE:', round(rf_rmse, 4))
print('MAE:', round(rf_mae, 4))
print('R^2:', round(rf_r2, 4))
print('MAPE:', round(rf_mape, 2), '%')
print('Training time (s):', round(rf_train_time, 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.scatterplot(x=y_test, y=rf_predictions, alpha=0.7, ax=axes[0])
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
axes[0].set_title('Predicted vs Actual Fit Score')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

residuals = y_test - rf_predictions
sns.scatterplot(x=rf_predictions, y=residuals, alpha=0.7, ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residual Plot')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')

sns.histplot(residuals, bins=20, kde=True, ax=axes[2], color='#4C72B0')
axes[2].set_title('Residual Distribution')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.show()

**Random Forest optimization summary:**

I optimized Random Forest through systematic hyperparameter experiments across tree count, maximum depth, and minimum leaf size. The tuned version improves predictive quality while keeping the model interpretable through feature importance, making it a strong final choice for a practical scouting support tool.


## 9. Training and Validation Diagnostics

In [ ]:
train_sizes, train_scores, validation_scores = learning_curve(
    estimator=clone(best_rf),
    X=X_train_processed,
    y=y_train,
    cv=5,
    scoring='neg_mean_absolute_error',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=1
)

train_mae = -train_scores.mean(axis=1)
validation_mae = -validation_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mae, marker='o', label='Training MAE')
plt.plot(train_sizes, validation_mae, marker='o', label='Validation MAE')
plt.title('Random Forest Learning Curve')
plt.xlabel('Training examples')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.show()

In [ ]:
feature_names = preprocessor.get_feature_names_out()
feature_importance = pd.DataFrame({'feature': feature_names, 'importance': best_rf.feature_importances_}).sort_values('importance', ascending=False)

display(feature_importance.head(15))

plt.figure(figsize=(9, 6))
top_features = feature_importance.head(12).copy()
sns.barplot(data=top_features, x='importance', y='feature', hue='feature', palette='Blues_r', dodge=False, legend=False)
plt.title('Top Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 10. Model Comparison and Example Predictions

In [ ]:
results = pd.DataFrame([
    {
        'Model': 'Optimized K-Means baseline',
        'Key hyperparameters': f"k={int(best_kmeans_params['n_clusters'])}, n_init={int(best_kmeans_params['n_init'])}",
        'RMSE': kmeans_rmse,
        'MAE': kmeans_mae,
        'R2': kmeans_r2,
        'MAPE': kmeans_mape,
        'Training time (s)': np.nan,
        'Strengths': 'Interpretable archetypes',
        'Limitations': 'Weak direct prediction precision'
    },
    {
        'Model': 'Optimized Random Forest',
        'Key hyperparameters': f"n_estimators={int(best_rf_params['n_estimators'])}, max_depth={int(best_rf_params['max_depth'])}, min_samples_leaf={int(best_rf_params['min_samples_leaf'])}",
        'RMSE': rf_rmse,
        'MAE': rf_mae,
        'R2': rf_r2,
        'MAPE': rf_mape,
        'Training time (s)': rf_train_time,
        'Strengths': 'Strong predictive accuracy',
        'Limitations': 'Less intuitive than cluster profiles'
    }
])

results_display = results.copy()
for col in ['RMSE', 'MAE', 'R2', 'MAPE', 'Training time (s)']:
    results_display[col] = results_display[col].round(4)
display(results_display)

In [ ]:
example_predictions = X_test.copy()
example_predictions['actual_fit_score'] = y_test.values
example_predictions['kmeans_pred'] = kmeans_predictions
example_predictions['rf_pred'] = rf_predictions
example_predictions['rf_error'] = example_predictions['actual_fit_score'] - example_predictions['rf_pred']
example_predictions['rf_abs_error'] = example_predictions['rf_error'].abs()
example_predictions['team_name'] = df.loc[X_test.index, 'team_name'].values
example_predictions['player_name'] = df.loc[X_test.index, 'player_name'].values

example_cols = [
    'player_name', 'team_name', 'position', 'season', 'goals_scored', 'assists', 'minutes', 'now_cost',
    'actual_fit_score', 'kmeans_pred', 'rf_pred', 'rf_error', 'rf_abs_error'
]

representative_examples = pd.concat([
    example_predictions.sort_values('rf_abs_error').head(3),
    example_predictions.sort_values('rf_abs_error', ascending=False).head(3)
]).drop_duplicates().reset_index(drop=True)

for col in ['actual_fit_score', 'kmeans_pred', 'rf_pred', 'rf_error', 'rf_abs_error']:
    representative_examples[col] = representative_examples[col].round(2)

display(representative_examples[example_cols])

**Performance comparison and model choice:**

Random Forest clearly outperforms the optimized K-Means baseline across every regression metric, especially RMSE, MAE, and R?. That pattern is consistent with the example predictions: K-Means can place players into meaningful stylistic buckets, but it struggles when two players belong to the same broad archetype while still having materially different Tottenham fit scores. Random Forest is better at capturing those more granular nonlinear differences.

The practical trade-off is that K-Means is easier to explain to non-technical stakeholders because its clusters resemble intuitive tactical archetypes, while Random Forest is more accurate but more complex. For deployment, I would recommend Random Forest as the primary prediction engine and keep K-Means as a supporting analysis tool for scouting conversations and profile validation.


## 11. Ethical Analysis & Responsible Deployment

This project has several potential sources of bias. The most important is representation bias: the draft dataset only covers Premier League players, so the model learns Tottenham fit from a narrow slice of the talent market and may underrate players from smaller leagues or different tactical environments. There is also measurement bias because the project uses FPL-style statistics and `now_cost` as a price proxy rather than true market value, wages, injury history, or richer tactical event data. That means the model may overvalue players who perform well in fantasy-friendly metrics while missing lower-profile players whose contributions are more tactical than statistical.

If the model makes poor predictions, the people most affected would be players who are incorrectly screened out, recruitment staff who waste time on misleading recommendations, and club decision-makers who may overpay or miss stronger fits. False positives could lead the club to pursue expensive players who do not fit the system, while false negatives could hide undervalued talent from consideration. Harm could also be uneven across positions or player backgrounds if the model systematically prefers players from stronger clubs, better-known teams, or more visible attacking roles.

Before any real deployment, I would mitigate these risks by expanding the dataset beyond the Premier League, auditing model performance by position and team context, replacing proxy value measures with better financial data, and keeping the model as a decision-support tool rather than a fully automated gatekeeper. Human scouting review should remain part of the process, especially for edge cases, younger players, and players from underrepresented leagues. Regular retraining and bias monitoring would also be needed as Tottenham's tactics and the transfer market evolve.


## 12. Business Recommendations & Deployment Considerations

The model suggests that Tottenham should prioritize recruitment profiles that combine strong influence, all-around contribution (`bps` and `bps_per90`), and efficient attacking output rather than relying only on raw goal totals. In practical terms, the club should use the ranking output to create a shortlist of players who combine tactical fit with manageable cost signals, then pass those candidates into a more detailed scouting process. Feature importance results indicate that broad contribution and involvement metrics matter more than simple box-score production, which aligns with a system-based recruitment approach.

For responsible deployment, I would not use the model as a fully automated selector. Instead, I would use it as a first-pass ranking tool that narrows the pool for analyst and scout review. The model should be retrained at least once per season or after a meaningful tactical shift, and performance should be monitored separately by position because defenders, midfielders, and forwards create value in different ways. A conservative deployment strategy would use the model to highlight candidates for review rather than reject players outright.

Stakeholders should also understand the model's limitations. It does not include injury history, personality, salary demands, league-adjusted event data, or true market value, so it should not be treated as a complete transfer-decision engine. It works best as a structured decision-support system for identifying interesting targets, not as a replacement for human judgment or full scouting context.


## 13. Final Reflection

This final version improves substantially on the earlier draft by adding more engineered features, systematic model optimization, richer diagnostics, example predictions, ethical analysis, and business recommendations. The next step beyond this coursework would be to replace the affordability proxy with Transfermarkt values, expand to multi-league player pools, and integrate richer FBref event data so the fit score becomes more faithful to the original proposal.
